## ***Water Consumption by Crops Data Cleaning***

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv('../Data/Raw Data/Water Consumption by Crops.csv')

In [3]:
df.head()

,Crop,Season,Year,Water_Consumption
0,Sugar Beet,Winter,2011,618118.0
1,Sugar Beet,Winter,2012,574576.0
2,Sugar Beet,Winter,2013,863085.0
3,Sugar Beet,Winter,2014,977227.0
4,Sugar Beet,Winter,2015,1020059.0


In [4]:
df.shape

(494, 4)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 494 entries, 0 to 493
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Crop               494 non-null    object 
 1   Season             494 non-null    object 
 2   Year               494 non-null    int64  
 3   Water_Consumption  469 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 15.6+ KB


In [6]:
df.describe(include='all')

,Crop,Season,Year,Water_Consumption
count,494,494,494.00000,4.690000e+02
unique,26,3,NaN,NaN
top,vegetables,Winter,NaN,NaN
freq,39,195,NaN,NaN
mean,NaN,NaN,2017.00000,9.341797e+05
std,NaN,NaN,3.74545,1.853114e+06
min,NaN,NaN,2011.00000,0.000000e+00
25%,NaN,NaN,2014.00000,9.477100e+03
50%,NaN,NaN,2017.00000,8.247900e+04
75%,NaN,NaN,2020.00000,8.630850e+05


In [7]:
df.columns

Index(['Crop', 'Season', 'Year', 'Water_Consumption'], dtype='object')

In [8]:
missing_values = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})

missing_values

,Missing_Count,Missing_Percentage
Crop,0,0.000000
Season,0,0.000000
Year,0,0.000000
Water_Consumption,25,5.060729


In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
missing_data = df[df['Water_Consumption'].isnull()]

missing_data.sort_values(
    by=['Crop', 'Season', 'Year']
)

,Crop,Season,Year,Water_Consumption
338,Alfalfa,Summer,2011,NaN
339,Alfalfa,Summer,2012,NaN
340,Alfalfa,Summer,2013,NaN
341,Alfalfa,Summer,2014,NaN
342,Alfalfa,Summer,2015,NaN
343,Alfalfa,Summer,2016,NaN
344,Alfalfa,Summer,2017,NaN
345,Alfalfa,Summer,2018,NaN
481,Green Fodder,Nile season,2011,NaN
482,Green Fodder,Nile season,2012,NaN


In [11]:
missing_data.groupby(
    ['Crop', 'Season']
).agg(
    Missing_Records=('Year', 'count'),
    Missing_Years=('Year', lambda x: sorted(x.tolist()))
).reset_index()

,Crop,Season,Missing_Records,Missing_Years
0,Alfalfa,Summer,8,"[2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]"
1,Green Fodder,Nile season,4,"[2011, 2012, 2013, 2014]"
2,"Medical, Aromatic Plant",Nile season,4,"[2011, 2012, 2013, 2014]"
3,Other varieties,Summer,1,[2019]
4,Other varieties,Winter,4,"[2019, 2021, 2022, 2023]"
5,Sesame,Nile season,4,"[2011, 2012, 2013, 2014]"


In [12]:
missing_analysis = df.groupby(
    ['Crop', 'Season']
).agg(
    Total_Records=('Year', 'count'),
    Available_Values=('Water_Consumption', 'count'),
    Missing_Values=('Water_Consumption', lambda x: x.isnull().sum()),
    First_Year=('Year', 'min'),
    Last_Year=('Year', 'max')
).reset_index()

missing_analysis['Missing_Percentage'] = (
    missing_analysis['Missing_Values'] /
    missing_analysis['Total_Records'] * 100
).round(2)

missing_analysis[
    missing_analysis['Missing_Values'] > 0
]

,Crop,Season,Total_Records,Available_Values,Missing_Values,First_Year,Last_Year,Missing_Percentage
0,Alfalfa,Summer,13,5,8,2011,2023,61.54
9,Green Fodder,Nile season,13,9,4,2011,2023,30.77
15,"Medical, Aromatic Plant",Nile season,13,9,4,2011,2023,30.77
21,Other varieties,Summer,13,12,1,2011,2023,7.69
22,Other varieties,Winter,13,9,4,2011,2023,30.77
25,Sesame,Nile season,13,9,4,2011,2023,30.77


In [13]:
for crop in missing_data['Crop'].unique():

    print(f"\nCrop: {crop}")

    display(
        df[df['Crop'] == crop]
        .groupby('Season')
        .agg(
            Total_Records=('Year', 'count'),
            Available_Values=('Water_Consumption', 'count'),
            Mean_Consumption=('Water_Consumption', 'mean')
        )
    )


Crop: Other varieties


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Summer,13,12,112574.758333
Winter,13,9,1754.000000



Crop: Alfalfa


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Summer,13,5,22796.0



Crop: Medical, Aromatic Plant


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Nile season,13,9,4724.888889
Summer,13,13,61503.838462
Winter,13,13,72963.846154



Crop: Sesame


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Nile season,13,9,4074.555556
Summer,13,13,133945.507692



Crop: Green Fodder


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Nile season,13,9,3.601322e+04
Summer,13,13,1.060641e+06


In [14]:
for crop in missing_data['Crop'].unique():

    print(f"\nCrop: {crop}")

    years = missing_data.loc[
        missing_data['Crop'] == crop, 'Year'
    ].unique()

    display(
        df[
            (df['Year'].isin(years)) &
            (df['Water_Consumption'].notna())
        ].groupby(['Year', 'Season'])
        ['Water_Consumption']
        .agg(['count', 'mean', 'min', 'max'])
    )


Crop: Other varieties


count          mean      min        max
Year Season                                              
2019 Nile season      8  7.754325e+04    527.0   400325.0
     Summer          14  1.680899e+06  10668.0  7987679.0
     Winter          14  8.608119e+05    377.0  5611016.0
2021 Nile season      8  5.059025e+04    240.0   297819.0
     Summer          15  1.481120e+06   5818.0  7884808.0
     Winter          14  8.742668e+05    250.0  6063736.0
2022 Nile season      8  5.824838e+04    199.0   292109.0
     Summer          15  1.482099e+06   2837.0  7120806.0
     Winter          14  7.825591e+05    199.0  5680058.0
2023 Nile season      8  4.019425e+04      0.0   147104.0
     Summer          15  1.518381e+06   6146.0  9043052.0
     Winter          14  8.009618e+05    250.0  4961025.0


Crop: Alfalfa


count          mean      min        max
Year Season                                              
2011 Nile season      5  2.422228e+05    748.0   759016.0
     Summer          14  1.316740e+06  22778.0  5717526.0
     Winter          15  6.206409e+05    809.0  4156133.0
2012 Nile season      5  2.342266e+05    162.0   714781.0
     Summer          14  1.189407e+06   5403.0  5520366.0
     Winter          15  6.014993e+05    815.0  4412625.0
2013 Nile season      5  2.312918e+05     13.0   689735.0
     Summer          14  1.571641e+06  16446.6  8660206.8
     Winter          15  7.501851e+05    587.0  5708154.0
2014 Nile season      5  2.243256e+05    443.0   618778.0
     Summer          14  1.660131e+06  16897.5  8504238.0
     Winter          15  7.608014e+05    131.0  5916342.0
2015 Nile season      8  1.194826e+05    330.0   557559.0
     Summer          14  1.474775e+06  16755.0  6154483.0
     Winter          15  7.599351e+05     83.0  5811857.0
2016 Nile season      8  1.438345e+05     73.0   645194.0
     Summer          14  1.841022e+06   8367.0  8041750.0
     Winter          15  8.490361e+05    557.0  6163666.0
2017 Nile season      8  1.346172e+05     73.0   584526.0
     Summer          14  1.851814e+06  10375.0  8343090.0
     Winter          15  7.332764e+05    312.0  4858120.0
2018 Nile season      8  1.007180e+05    187.0   419817.0
     Summer          14  1.490027e+06  12088.0  7444348.0
     Winter          15  7.221597e+05    147.0  5367136.0


Crop: Medical, Aromatic Plant


count          mean      min        max
Year Season                                              
2011 Nile season      5  2.422228e+05    748.0   759016.0
     Summer          14  1.316740e+06  22778.0  5717526.0
     Winter          15  6.206409e+05    809.0  4156133.0
2012 Nile season      5  2.342266e+05    162.0   714781.0
     Summer          14  1.189407e+06   5403.0  5520366.0
     Winter          15  6.014993e+05    815.0  4412625.0
2013 Nile season      5  2.312918e+05     13.0   689735.0
     Summer          14  1.571641e+06  16446.6  8660206.8
     Winter          15  7.501851e+05    587.0  5708154.0
2014 Nile season      5  2.243256e+05    443.0   618778.0
     Summer          14  1.660131e+06  16897.5  8504238.0
     Winter          15  7.608014e+05    131.0  5916342.0


Crop: Sesame


count          mean      min        max
Year Season                                              
2011 Nile season      5  2.422228e+05    748.0   759016.0
     Summer          14  1.316740e+06  22778.0  5717526.0
     Winter          15  6.206409e+05    809.0  4156133.0
2012 Nile season      5  2.342266e+05    162.0   714781.0
     Summer          14  1.189407e+06   5403.0  5520366.0
     Winter          15  6.014993e+05    815.0  4412625.0
2013 Nile season      5  2.312918e+05     13.0   689735.0
     Summer          14  1.571641e+06  16446.6  8660206.8
     Winter          15  7.501851e+05    587.0  5708154.0
2014 Nile season      5  2.243256e+05    443.0   618778.0
     Summer          14  1.660131e+06  16897.5  8504238.0
     Winter          15  7.608014e+05    131.0  5916342.0


Crop: Green Fodder


count          mean      min        max
Year Season                                              
2011 Nile season      5  2.422228e+05    748.0   759016.0
     Summer          14  1.316740e+06  22778.0  5717526.0
     Winter          15  6.206409e+05    809.0  4156133.0
2012 Nile season      5  2.342266e+05    162.0   714781.0
     Summer          14  1.189407e+06   5403.0  5520366.0
     Winter          15  6.014993e+05    815.0  4412625.0
2013 Nile season      5  2.312918e+05     13.0   689735.0
     Summer          14  1.571641e+06  16446.6  8660206.8
     Winter          15  7.501851e+05    587.0  5708154.0
2014 Nile season      5  2.243256e+05    443.0   618778.0
     Summer          14  1.660131e+06  16897.5  8504238.0
     Winter          15  7.608014e+05    131.0  5916342.0

In [15]:
df_clean = df.dropna(
    subset=['Water_Consumption'])

In [16]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 469 entries, 0 to 493
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Crop               469 non-null    object 
 1   Season             469 non-null    object 
 2   Year               469 non-null    int64  
 3   Water_Consumption  469 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 18.3+ KB


In [17]:
df_clean.to_csv("../Data/Clean Data/Water_Consumption_Crops_clean.csv", index=False)